# Lesson 3: Agentic Search

### 本节课的核心思路：什么是「Agentic Search」，以及无密钥的替代方案

「Agentic Search」指的是专门为 LLM 设计的智能搜索（如 Tavily）：它不只是返回一堆链接，
还能直接给出结构化、精简过的答案，省去 Agent 自己抓网页、清洗 HTML 的麻烦。

不过 Tavily 需要申请 API 密钥。本课改用**完全无需密钥**的方案来演示同一个对比：
1. 用普通的 DuckDuckGo 搜索 + `requests` + `BeautifulSoup` 手动抓取、清洗网页内容（传统爬虫方式，噪音多、步骤繁琐）；
2. 再直接看 DuckDuckGo 返回的结构化结果（title/href/body），体会「拿到相对干净的信息」省了多少事。

要点：如果你想要 Tavily 那种「一句话总结」的效果，可以把 DuckDuckGo 返回的若干条 body
再交给本地 LLM（前两课的 Ollama+Qwen）去提炼——这才是"agentic"的本质：搜索 + 模型协作。

In [ ]:
from dotenv import load_dotenv
import os
_ = load_dotenv()  # 本课改用无密钥的 DuckDuckGo，不再需要 TAVILY_API_KEY
# 【改为无密钥搜索】原课程用 Tavily（需 TAVILY_API_KEY）。这里统一用 DuckDuckGo（无需任何密钥）。
from duckduckgo_search import DDGS
ddg = DDGS()  # 构造客户端本身不发请求，第一次调用 .text() 时才真正联网搜索

In [ ]:
# 用 DuckDuckGo 搜索（无密钥）。注意：它不像 Tavily 那样直接给"一句话答案"，
# 而是返回若干条 {title, href, body} 结果；若想要"一句话总结"，可把这些 body 交给本地 LLM 提炼。
results = ddg.text("英伟达新的 Blackwell GPU 有什么特点？", max_results=4)
for r in results:
    print("-", r["title"])
    print("  ", r["body"][:200])

In [ ]:
# 选择一个城市（可以改成你自己的城市试试！）
city = "北京"

query = f"""
    {city} 今天的天气怎么样？
    我今天适合出行吗？
    "weather.com"
"""

In [ ]:
import requests
from bs4 import BeautifulSoup
# 提示：duckduckgo_search 这个包已改名为 ddgs（pip install ddgs），当前仍可用，只会有 RuntimeWarning，不影响运行
from duckduckgo_search import DDGS
import re

ddg=DDGS()
def search(query,max_results=6):
    try:
        results=ddg.text(query,max_results=max_results)  # 普通搜索：只返回 title/href/body，没有 Tavily 那种"直接给答案"的能力
        return [i ["href"] for i in results]              # 从结果列表里只取每条结果的链接
    except Exception as e:
        print(f"returning previous results due to exception reaching ddg.")
        results = [ # cover case where DDG rate limits due to high deeplearning.ai volume
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results
for i in search(query):
    print(i)

In [ ]:
def scrape_weather_info(url):
    """Scrape content from the given URL"""
    if not url:
        return "Weather information could not be found."

    # fetch data
    headers={'User-Agent': 'Mozilla/5.0'}   # 伪装成浏览器 UA，避免部分网站直接拒绝无 UA 的请求
    response=requests.get(url,headers=headers)
    if response.status_code!=200:
        return "Failed to retrieve the webpage."
    soup=BeautifulSoup(response.text,'html.parser')   # 用 BeautifulSoup 把原始 HTML 解析成可查询的 DOM 树
    return soup

In [ ]:
# use DuckDuckGo to find websites and take the first result
url = search(query)[0]

# scrape first wesbsite
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:50000]) # limit long outputs  # 抓下来的是原始 HTML（含大量标签、样式、脚本噪音），下一步才会清洗成纯文本

In [ ]:
weather_data=[]
for tag in soup.find_all(['h1','h2','h3','p']):   # 只挑标题和段落标签，过滤掉 script/style/nav 等噪音标签
    text=tag.get_text(" ",strip=True)
    weather_data.append(text)
weather_data="\n".join(weather_data)
weather_data=re.sub(r'\s+',' ',weather_data)      # 把多个连续空白字符压缩成一个空格，让文本更紧凑
print(f"Website: {url}\n\n")
print(weather_data)
# 对比一下：光是拿到"干净文本"就需要 请求网页 -> 解析 HTML -> 按标签过滤 -> 正则清洗 这好几步，
# 而且不同网站的 HTML 结构完全不同，这段代码换个网站可能就要重写。这正是下面 Tavily 一行代码要解决的问题。

In [ ]:
# 用 DuckDuckGo 搜同一个问题，取第一条结果的正文，
# 对比上面「手动请求网页 -> 解析 HTML -> 按标签过滤 -> 正则清洗」那一大串步骤
results = ddg.text(query, max_results=1)
data = results[0]["body"] if results else "（未获取到搜索结果）"
print(data)

In [ ]:
import json
from pygments import highlight, lexers, formatters
# DuckDuckGo 返回的本身就是结构化的 dict 列表（title/href/body），可直接美化打印，
# 无需像原 Tavily 代码那样再把字符串 replace 引号后 json.loads（那种做法遇到文本里的撇号就会崩）。
results = ddg.text(query, max_results=3)
formatted_json = json.dumps(results, indent=4, ensure_ascii=False)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())  # 终端高亮打印 JSON，方便阅读结构化结果
print(colorful_json)